# 02 PLS Operational Modeling: Single-Dataset Moisture Prediction

1つの実データCSVを `train / valid / test` に分割し、PLS回帰だけで含水率モデルを構築する標準運用 notebook です。複数モデル比較・アンサンブル・特徴量選択などの本格検討は `03_advanced_model_selection_ensemble_production.ipynb` に集約します。

この notebook の役割は、前処理条件とPLS成分数を確認し、説明しやすく再現しやすい単一PLSモデルを保存することです。

## 0. 設定と共通関数

In [ ]:

from __future__ import annotations

from pathlib import Path
import json
import math
import os
import pickle
import re
import warnings

import numpy as np
import pandas as pd
os.environ.setdefault('MPLCONFIGDIR', str(Path('/private/tmp') / 'matplotlib-cache'))
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import savgol_filter, find_peaks
from sklearn.base import BaseEstimator, TransformerMixin, RegressorMixin, clone
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, GroupShuffleSplit, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.validation import check_array, check_is_fitted

warnings.filterwarnings('ignore', category=ConvergenceWarning)
warnings.filterwarnings('ignore', category=UserWarning)
sns.set_theme(style='whitegrid', context='notebook')

for font_name in ['Hiragino Sans', 'AppleGothic', 'Apple SD Gothic Neo', 'Yu Gothic', 'Meiryo', 'Noto Sans CJK JP']:
    try:
        plt.rcParams['font.family'] = font_name
        break
    except Exception:
        pass
plt.rcParams['axes.unicode_minus'] = False

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = Path('/Users/ogawatomohiro/myproject')
DATA_DIR = PROJECT_ROOT / 'data'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'production'
FIGURE_DIR = OUTPUT_DIR / 'figures'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# 実運用ではここを実データCSVに変更してください。存在しない場合だけサンプルの train.csv で動作確認します。
DATA_PATH = DATA_DIR / 'production_dataset.csv'
SAMPLE_FALLBACK_PATH = DATA_DIR / 'train.csv'
ENCODINGS = ('cp932', 'utf-8-sig', 'utf-8', 'shift_jis')

TARGET_CANDIDATES = ['含水率', 'mc', 'moisture', 'moisture_content', 'water_content', 'target', 'y']
SAMPLE_CANDIDATES = ['サンプル名', 'サンプルID', 'sample name', 'sample_name', 'sample number', 'sample_id', 'id', 'ID']
DATE_CANDIDATES = ['日付', '測定日', 'date', 'measurement_date', 'measured_at']
GROUP_CANDIDATES = ['ロット', 'lot', 'batch', '樹種', 'species', 'species number', 'sample_name']

# 波長列は 900-1700 nm を主対象にします。実データが範囲外まで含む場合は None にしてください。
WAVELENGTH_RANGE_NM = (900, 1700)
VALID_SIZE = 0.20
TEST_SIZE = 0.20
RANDOM_STATE = 42
N_SPLITS = 5
GROUP_SPLIT_COL = None  # 例: '日付' や 'ロット'。None の場合は目的変数の分布を保つ通常split。

# PLS運用設定。AUTO_SELECT_PLS=TrueならCV/validで自動選択、Falseなら下の指定値を使います。
AUTO_SELECT_PLS = True
MANUAL_PREPROCESSING = 'savgol_2nd_21'  # raw, savgol_smooth_21, savgol_1st_21, savgol_2nd_21, savgol_2nd_41, snv_savgol_2nd_21, msc_savgol_2nd_21
MANUAL_N_COMPONENTS = 10


def read_csv_with_fallback(path: Path, encodings=ENCODINGS) -> pd.DataFrame:
    last_error = None
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError as exc:
            last_error = exc
    raise last_error


def pick_first_existing(columns, candidates, required=False, label='column'):
    normalized = {str(c).strip().lower(): c for c in columns}
    for cand in candidates:
        key = str(cand).strip().lower()
        if key in normalized:
            return normalized[key]
    if required:
        raise ValueError(f'{label} not found. candidates={candidates}')
    return None


def parse_wavelength_from_col(col):
    text = str(col).strip()
    if text in {'', 'nan'}:
        return None
    # Examples: 900, 900.0, 900nm, wl_900, Abs_900, absorbance_900_nm
    nums = re.findall(r'(?<!\d)(\d{3,5}(?:\.\d+)?)(?!\d)', text)
    if not nums:
        return None
    values = [float(x) for x in nums]
    # Prefer values in the expected nm range when multiple numbers exist.
    lo, hi = WAVELENGTH_RANGE_NM if WAVELENGTH_RANGE_NM is not None else (0, float('inf'))
    in_range = [v for v in values if lo <= v <= hi]
    return in_range[-1] if in_range else values[-1]


def detect_spectral_columns(df: pd.DataFrame, exclude_cols=None, wavelength_range_nm=WAVELENGTH_RANGE_NM):
    exclude = set(exclude_cols or [])
    records = []
    for col in df.columns:
        if col in exclude:
            continue
        wl = parse_wavelength_from_col(col)
        if wl is None:
            continue
        numeric = pd.to_numeric(df[col], errors='coerce')
        non_na_ratio = numeric.notna().mean()
        if non_na_ratio < 0.90:
            continue
        records.append((col, wl))
    if not records:
        raise ValueError('波長列を検出できませんでした。列名に 900, 901nm, Abs_900 などの波長値が含まれているか確認してください。')
    spec_info = pd.DataFrame(records, columns=['column', 'wavelength_nm']).drop_duplicates('column')
    if wavelength_range_nm is not None:
        lo, hi = wavelength_range_nm
        ranged = spec_info.query('@lo <= wavelength_nm <= @hi').copy()
        if len(ranged) >= 10:
            spec_info = ranged
    spec_info = spec_info.sort_values('wavelength_nm').reset_index(drop=True)
    return spec_info['column'].tolist(), spec_info


def make_y_bins(y, max_bins=8):
    y = pd.Series(y).astype(float)
    n_bins = min(max_bins, max(2, len(y) // 10))
    try:
        bins = pd.qcut(y, q=n_bins, duplicates='drop')
        counts = bins.value_counts()
        if len(counts) >= 2 and counts.min() >= 2:
            return bins.astype(str)
    except Exception:
        pass
    return None


def split_single_dataset(df, target_col, group_col=None, valid_size=VALID_SIZE, test_size=TEST_SIZE, random_state=RANDOM_STATE):
    df = df.dropna(subset=[target_col]).reset_index(drop=True).copy()
    if group_col is not None and group_col in df.columns and df[group_col].nunique() >= 3:
        splitter1 = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
        train_valid_idx, test_idx = next(splitter1.split(df, groups=df[group_col]))
        train_valid = df.iloc[train_valid_idx].reset_index(drop=True)
        test_df = df.iloc[test_idx].reset_index(drop=True)
        rel_valid = valid_size / (1 - test_size)
        splitter2 = GroupShuffleSplit(n_splits=1, test_size=rel_valid, random_state=random_state)
        train_idx, valid_idx = next(splitter2.split(train_valid, groups=train_valid[group_col]))
        return train_valid.iloc[train_idx].reset_index(drop=True), train_valid.iloc[valid_idx].reset_index(drop=True), test_df

    stratify = make_y_bins(df[target_col])
    train_valid, test_df = train_test_split(df, test_size=test_size, random_state=random_state, shuffle=True, stratify=stratify)
    rel_valid = valid_size / (1 - test_size)
    stratify_tv = make_y_bins(train_valid[target_col])
    train_df, valid_df = train_test_split(train_valid, test_size=rel_valid, random_state=random_state, shuffle=True, stratify=stratify_tv)
    return train_df.reset_index(drop=True), valid_df.reset_index(drop=True), test_df.reset_index(drop=True)


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def regression_metrics(y_true, y_pred):
    return {
        'r2': float(r2_score(y_true, y_pred)),
        'rmse': rmse(y_true, y_pred),
        'mae': float(mean_absolute_error(y_true, y_pred)),
    }


def odd_window(window_length, n_points):
    w = int(window_length)
    if w % 2 == 0:
        w += 1
    max_w = n_points if n_points % 2 == 1 else n_points - 1
    w = max(3, min(w, max_w))
    return w


def load_production_dataset():
    path = DATA_PATH if DATA_PATH.exists() else SAMPLE_FALLBACK_PATH
    if path == SAMPLE_FALLBACK_PATH:
        print(f'NOTE: {DATA_PATH} がないため、動作確認用に {SAMPLE_FALLBACK_PATH} を読み込みます。')
    df = read_csv_with_fallback(path)
    target_col = pick_first_existing(df.columns, TARGET_CANDIDATES, required=True, label='target column')
    sample_col = pick_first_existing(df.columns, SAMPLE_CANDIDATES, required=False, label='sample id column')
    date_col = pick_first_existing(df.columns, DATE_CANDIDATES, required=False, label='date column')
    group_col = GROUP_SPLIT_COL or pick_first_existing(df.columns, GROUP_CANDIDATES, required=False, label='diagnostic group column')
    exclude_cols = [c for c in [target_col, sample_col, date_col, group_col] if c is not None]
    spectral_cols, spec_info = detect_spectral_columns(df, exclude_cols=exclude_cols)
    df[spectral_cols] = df[spectral_cols].apply(pd.to_numeric, errors='coerce')
    if date_col is not None:
        df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    train_df, valid_df, test_df = split_single_dataset(df, target_col=target_col, group_col=GROUP_SPLIT_COL)
    config = {
        'data_path': str(path),
        'target_col': target_col,
        'sample_col': sample_col,
        'date_col': date_col,
        'diagnostic_group_col': group_col,
        'group_split_col': GROUP_SPLIT_COL,
        'n_spectral_cols': len(spectral_cols),
        'wavelength_min_nm': float(spec_info['wavelength_nm'].min()),
        'wavelength_max_nm': float(spec_info['wavelength_nm'].max()),
        'train_shape': train_df.shape,
        'valid_shape': valid_df.shape,
        'test_shape': test_df.shape,
    }
    return df, train_df, valid_df, test_df, spectral_cols, spec_info, config


def get_xy(part_df, spectral_cols, target_col):
    return part_df[spectral_cols].astype(float), part_df[target_col].astype(float)


def sample_ids(df, sample_col):
    if sample_col is not None and sample_col in df.columns:
        return df[sample_col].values
    return np.arange(len(df))


class SavitzkyGolayTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, window_length=21, polyorder=2, deriv=0, mode='interp'):
        self.window_length = window_length
        self.polyorder = polyorder
        self.deriv = deriv
        self.mode = mode

    def fit(self, X, y=None):
        X = check_array(X, dtype=float)
        self.n_features_in_ = X.shape[1]
        self.window_length_ = odd_window(self.window_length, self.n_features_in_)
        self.polyorder_ = min(int(self.polyorder), self.window_length_ - 1)
        return self

    def transform(self, X):
        check_is_fitted(self, ['n_features_in_', 'window_length_', 'polyorder_'])
        X = check_array(X, dtype=float)
        return savgol_filter(X, window_length=self.window_length_, polyorder=self.polyorder_, deriv=self.deriv, axis=1, mode=self.mode)


class SNVTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        X = check_array(X, dtype=float)
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X):
        check_is_fitted(self, ['n_features_in_'])
        X = check_array(X, dtype=float)
        mean = X.mean(axis=1, keepdims=True)
        std = X.std(axis=1, keepdims=True)
        std[std == 0] = 1.0
        return (X - mean) / std


class MSCTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        X = check_array(X, dtype=float)
        self.n_features_in_ = X.shape[1]
        self.reference_ = X.mean(axis=0)
        return self

    def transform(self, X):
        check_is_fitted(self, ['reference_'])
        X = check_array(X, dtype=float)
        corrected = np.empty_like(X, dtype=float)
        ref = self.reference_
        for i, row in enumerate(X):
            slope, intercept = np.polyfit(ref, row, deg=1)
            if abs(slope) < 1e-12:
                corrected[i] = row - intercept
            else:
                corrected[i] = (row - intercept) / slope
        return corrected


class AdaptivePLSRegression(BaseEstimator, RegressorMixin):
    def __init__(self, n_components=10, scale=True, max_iter=500, tol=1e-06):
        self.n_components = n_components
        self.scale = scale
        self.max_iter = max_iter
        self.tol = tol

    def fit(self, X, y):
        X = check_array(X, dtype=float)
        y = np.asarray(y, dtype=float)
        n_components = max(1, min(int(self.n_components), X.shape[0] - 1, X.shape[1]))
        self.n_components_ = n_components
        self.model_ = PLSRegression(n_components=n_components, scale=self.scale, max_iter=self.max_iter, tol=self.tol)
        self.model_.fit(X, y)
        return self

    def predict(self, X):
        check_is_fitted(self, ['model_'])
        return self.model_.predict(X).ravel()


## 1. データ読み込みと split

In [ ]:
df, train_df, valid_df, test_df, spectral_cols, spec_info, config = load_production_dataset()
print(json.dumps(config, ensure_ascii=False, indent=2, default=str))

X_train, y_train = get_xy(train_df, spectral_cols, config['target_col'])
X_valid, y_valid = get_xy(valid_df, spectral_cols, config['target_col'])
X_test, y_test = get_xy(test_df, spectral_cols, config['target_col'])
wavelengths = spec_info['wavelength_nm'].to_numpy(float)
sample_col = config['sample_col']
group_col = config['diagnostic_group_col']

print(X_train.shape, X_valid.shape, X_test.shape)

## 2. PLS成分数と前処理条件の探索

train 内のKFold CVで、PLS専用の前処理候補と成分数だけを評価します。`AUTO_SELECT_PLS = False` にすると、冒頭で指定した `MANUAL_PREPROCESSING` と `MANUAL_N_COMPONENTS` をそのまま採用できます。

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
def make_cv(n_splits=N_SPLITS):
    return KFold(n_splits=min(n_splits, len(X_train)), shuffle=True, random_state=RANDOM_STATE)

def evaluate_candidate_cv(estimator, name):
    cv = make_cv()
    pred = cross_val_predict(estimator, X_train, y_train, cv=cv)
    out = {'model_name': name, **regression_metrics(y_train, pred)}
    return out, pred

preprocessing_candidates = {
    'raw': [],
    'savgol_smooth_21': [('savgol', SavitzkyGolayTransformer(window_length=21, polyorder=2, deriv=0))],
    'savgol_1st_21': [('savgol', SavitzkyGolayTransformer(window_length=21, polyorder=2, deriv=1))],
    'savgol_2nd_21': [('savgol', SavitzkyGolayTransformer(window_length=21, polyorder=2, deriv=2))],
    'savgol_2nd_41': [('savgol', SavitzkyGolayTransformer(window_length=41, polyorder=2, deriv=2))],
    'snv_savgol_2nd_21': [('snv', SNVTransformer()), ('savgol', SavitzkyGolayTransformer(window_length=21, polyorder=2, deriv=2))],
    'msc_savgol_2nd_21': [('msc', MSCTransformer()), ('savgol', SavitzkyGolayTransformer(window_length=21, polyorder=2, deriv=2))],
}

def build_pls_pipeline(prep_name, n_components):
    if prep_name not in preprocessing_candidates:
        raise KeyError(f'Unknown preprocessing: {prep_name}. Choose from {list(preprocessing_candidates)}')
    return Pipeline(preprocessing_candidates[prep_name] + [('pls', AdaptivePLSRegression(n_components=n_components, scale=True))])

if not AUTO_SELECT_PLS:
    if MANUAL_PREPROCESSING not in preprocessing_candidates:
        raise KeyError(f'MANUAL_PREPROCESSING={MANUAL_PREPROCESSING} is not defined.')
    if int(MANUAL_N_COMPONENTS) < 1:
        raise ValueError('MANUAL_N_COMPONENTS must be >= 1.')

cv_rows = []
oof_frames = []
component_grid = [2, 3, 5, 8, 10, 12, 15, 20]
for prep_name in preprocessing_candidates:
    for n_comp in component_grid:
        est = build_pls_pipeline(prep_name, n_comp)
        name = f'{prep_name}__pls{n_comp}'
        try:
            row, pred = evaluate_candidate_cv(est, name)
            row.update({'preprocessing': prep_name, 'n_components': n_comp})
            cv_rows.append(row)
            oof_frames.append(pd.DataFrame({'sample_id': sample_ids(train_df, sample_col), 'model_name': name, 'y_true': y_train.values, 'oof_pred': pred}))
        except Exception as exc:
            cv_rows.append({'model_name': name, 'preprocessing': prep_name, 'n_components': n_comp, 'r2': np.nan, 'rmse': np.nan, 'mae': np.nan, 'error': repr(exc)})

cv_scores = pd.DataFrame(cv_rows).sort_values('rmse')
oof_df = pd.concat(oof_frames, ignore_index=True) if oof_frames else pd.DataFrame()
cv_scores.to_csv(OUTPUT_DIR / '02_pls_operational_cv_scores.csv', index=False, encoding='utf-8-sig')
oof_df.to_csv(OUTPUT_DIR / '02_pls_operational_oof_predictions.csv', index=False, encoding='utf-8-sig')
display(cv_scores.head(15))


## 3. PLS主成分数の解釈

選択対象の前処理について、主成分数を増やしたときの `R2` と説明分散比を確認します。データセットが少数成分で説明できるのか、成分を増やすと過学習しやすいのかを見るための診断です。

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
interpret_prep = MANUAL_PREPROCESSING if not AUTO_SELECT_PLS else cv_scores.iloc[0]['preprocessing']
max_components_for_curve = min(20, X_train.shape[0] - 1, X_train.shape[1])
curve_rows = []
for n_comp in range(1, max_components_for_curve + 1):
    est = build_pls_pipeline(interpret_prep, n_comp)
    cv_pred = cross_val_predict(est, X_train, y_train, cv=make_cv())
    est.fit(X_train, y_train)
    pls_model = est.named_steps['pls'].model_
    x_score_var = np.var(pls_model.x_scores_, axis=0, ddof=0)
    y_score_var = np.var(pls_model.y_scores_, axis=0, ddof=0)
    x_total_var = np.var(np.asarray(X_train, dtype=float), axis=0, ddof=0).sum()
    y_total_var = np.var(np.asarray(y_train, dtype=float), ddof=0)
    curve_rows.append({
        'preprocessing': interpret_prep,
        'n_components': n_comp,
        'cv_r2': r2_score(y_train, cv_pred),
        'cv_rmse': rmse(y_train, cv_pred),
        'train_r2': est.score(X_train, y_train),
        'x_explained_variance_cumulative': float(x_score_var[:n_comp].sum() / x_total_var) if x_total_var > 0 else np.nan,
        'y_score_variance_cumulative': float(y_score_var[:n_comp].sum() / y_total_var) if y_total_var > 0 else np.nan,
    })
component_curve_df = pd.DataFrame(curve_rows)
component_curve_df.to_csv(OUTPUT_DIR / '02_pls_operational_component_curve.csv', index=False, encoding='utf-8-sig')

display(component_curve_df.head())
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(component_curve_df['n_components'], component_curve_df['x_explained_variance_cumulative'], marker='o', label='X score variance')
axes[0].plot(component_curve_df['n_components'], component_curve_df['y_score_variance_cumulative'], marker='o', label='Y score variance')
axes[0].set_xlabel('Number of PLS components')
axes[0].set_ylabel('Cumulative variance ratio')
axes[0].set_title('PLS components vs explained variance')
axes[0].legend()
axes[1].plot(component_curve_df['n_components'], component_curve_df['train_r2'], marker='o', label='Train R2')
axes[1].plot(component_curve_df['n_components'], component_curve_df['cv_r2'], marker='o', label='CV R2')
axes[1].set_xlabel('Number of PLS components')
axes[1].set_ylabel('R2 score')
axes[1].set_title('PLS components vs R2 score')
axes[1].legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / '02_pls_operational_component_diagnostics.png', dpi=160)


## 4. valid で候補確認

自動選択時はCV上位PLSをvalidで確認します。手動指定時も同じ表に手動構成のvalid結果を出し、後段では手動構成を採用します。

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
valid_rows = []
if AUTO_SELECT_PLS:
    candidate_iter = cv_scores.dropna(subset=['rmse']).head(10).iterrows()
else:
    manual_row = pd.Series({'model_name': f'{MANUAL_PREPROCESSING}__pls{int(MANUAL_N_COMPONENTS)}', 'preprocessing': MANUAL_PREPROCESSING, 'n_components': int(MANUAL_N_COMPONENTS), 'rmse': np.nan})
    candidate_iter = [(0, manual_row)]

for _, row in candidate_iter:
    est = build_pls_pipeline(row['preprocessing'], int(row['n_components']))
    est.fit(X_train, y_train)
    pred = est.predict(X_valid)
    valid_rows.append({'model_name': row['model_name'], 'cv_rmse': row.get('rmse', np.nan), **regression_metrics(y_valid, pred)})
valid_scores = pd.DataFrame(valid_rows).sort_values('rmse')
valid_scores.to_csv(OUTPUT_DIR / '02_pls_operational_valid_scores.csv', index=False, encoding='utf-8-sig')
display(valid_scores)

if AUTO_SELECT_PLS:
    best_name = valid_scores.iloc[0]['model_name']
    best_row = cv_scores.loc[cv_scores['model_name'] == best_name].iloc[0]
else:
    best_name = f'{MANUAL_PREPROCESSING}__pls{int(MANUAL_N_COMPONENTS)}'
    best_row = pd.Series({'model_name': best_name, 'preprocessing': MANUAL_PREPROCESSING, 'n_components': int(MANUAL_N_COMPONENTS), 'rmse': np.nan})
best_estimator = build_pls_pipeline(best_row['preprocessing'], int(best_row['n_components']))
print('selected:', best_name)


## 5. PLS最終候補の確定

valid RMSE が最も小さい PLS 候補を採用します。運用版として単一モデルに固定し、比較対象モデルは作りません。

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
selected_model_name = valid_scores.iloc[0]['model_name']
selected_row = cv_scores.loc[cv_scores['model_name'] == selected_model_name].iloc[0]
selected_preprocessing = selected_row['preprocessing']
selected_n_components = int(selected_row['n_components'])
selected_valid_metrics = valid_scores.iloc[0].to_dict()

selected_summary = pd.DataFrame([
    {
        'selected_model': selected_model_name,
        'preprocessing': selected_preprocessing,
        'n_components': selected_n_components,
        'cv_rmse': float(selected_row['rmse']),
        'valid_r2': float(selected_valid_metrics['r2']),
        'valid_rmse': float(selected_valid_metrics['rmse']),
        'valid_mae': float(selected_valid_metrics['mae']),
    }
])
selected_summary.to_csv(OUTPUT_DIR / '02_pls_operational_selected_model_valid.csv', index=False, encoding='utf-8-sig')
display(selected_summary)

selected_pls = build_pls_pipeline(selected_preprocessing, selected_n_components)
selected_pls.fit(X_train, y_train)
valid_pred = selected_pls.predict(X_valid)

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(y_valid, valid_pred, s=28, alpha=0.75)
low = min(y_valid.min(), valid_pred.min())
high = max(y_valid.max(), valid_pred.max())
pad = (high - low) * 0.05 if high > low else 1
ax.plot([low - pad, high + pad], [low - pad, high + pad], color='black', linestyle='--')
ax.set_xlim(low - pad, high + pad)
ax.set_ylim(low - pad, high + pad)
ax.set_xlabel('Measured moisture')
ax.set_ylabel('Predicted moisture')
ax.set_title(f'Valid evaluation: {selected_model_name}')
ax.text(0.04, 0.96, f"R2={selected_valid_metrics['r2']:.3f}\nRMSE={selected_valid_metrics['rmse']:.3f}\nMAE={selected_valid_metrics['mae']:.3f}", transform=ax.transAxes, va='top', bbox={'facecolor': 'white', 'edgecolor': '#cccccc'})
fig.tight_layout()
fig.savefig(FIGURE_DIR / '02_pls_operational_valid_selected_model.png', dpi=160)


## 6. test 最終評価と保存

確定したPLS構成を `train + valid` で再学習し、最後に `test` で独立評価します。保存されるモデルもPLS単一モデルです。

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
final_estimator = build_pls_pipeline(selected_preprocessing, selected_n_components)
train_valid_df = pd.concat([train_df, valid_df], ignore_index=True)
X_train_valid, y_train_valid = get_xy(train_valid_df, spectral_cols, config['target_col'])
final_estimator.fit(X_train_valid, y_train_valid)
test_pred = final_estimator.predict(X_test)
test_metrics = regression_metrics(y_test, test_pred)
print(json.dumps({'final_model': selected_model_name, **test_metrics}, ensure_ascii=False, indent=2))

prediction_df = pd.DataFrame({
    'sample_id': sample_ids(test_df, sample_col),
    'split': 'test',
    'measured_moisture': y_test.values,
    'predicted_moisture': test_pred,
    'residual': y_test.values - test_pred,
})
if group_col is not None and group_col in test_df.columns:
    prediction_df['diagnostic_group'] = test_df[group_col].astype(str).values
prediction_df.to_csv(OUTPUT_DIR / '02_pls_operational_test_predictions.csv', index=False, encoding='utf-8-sig')
with open(OUTPUT_DIR / '02_pls_operational_final_model.pkl', 'wb') as f:
    pickle.dump({'model': final_estimator, 'spectral_cols': spectral_cols, 'config': config, 'spec_info': spec_info, 'selected_model_name': selected_model_name}, f)

train_valid_pred = final_estimator.predict(X_train_valid)
valid_eval_estimator = build_pls_pipeline(selected_preprocessing, selected_n_components).fit(X_train, y_train)
valid_pred = valid_eval_estimator.predict(X_valid)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
plot_sets = [
    ('Train+Valid fit', y_train_valid, train_valid_pred),
    ('Valid check', y_valid, valid_pred),
    ('Final test', y_test, test_pred),
]
all_true = np.concatenate([np.asarray(yv, dtype=float) for _, yv, _ in plot_sets])
all_pred = np.concatenate([np.asarray(pv, dtype=float) for _, _, pv in plot_sets])
low = min(all_true.min(), all_pred.min())
high = max(all_true.max(), all_pred.max())
pad = (high - low) * 0.05 if high > low else 1
for ax, (title, y_true_plot, y_pred_plot) in zip(axes, plot_sets):
    m = regression_metrics(y_true_plot, y_pred_plot)
    ax.scatter(y_true_plot, y_pred_plot, s=26, alpha=0.75)
    ax.plot([low - pad, high + pad], [low - pad, high + pad], color='black', linestyle='--', linewidth=1)
    ax.set_xlim(low - pad, high + pad)
    ax.set_ylim(low - pad, high + pad)
    ax.set_xlabel('Measured moisture')
    ax.set_ylabel('Predicted moisture')
    ax.set_title(title)
    ax.text(0.04, 0.96, f"R2={m['r2']:.3f}\nRMSE={m['rmse']:.3f}\nMAE={m['mae']:.3f}", transform=ax.transAxes, va='top', bbox={'facecolor': 'white', 'edgecolor': '#cccccc'})
fig.suptitle(f'PLS measurement accuracy: {selected_model_name}', y=1.02)
fig.tight_layout()
fig.savefig(FIGURE_DIR / '02_pls_operational_measurement_accuracy_scatter.png', dpi=160)
display(prediction_df.head())
